In [1]:
import torch.nn as nn
import torch
from mpramnist.BarbadillaMartinez2026 import BarbadillaMartinezDataset
from mpramnist.BarbadillaMartinez2026 import LitModel_BarbadillaMartinez
from mpramnist.models import HumanLegNet
import mpramnist.transforms as t
from torch.utils.data import DataLoader
from scipy.stats import pearsonr

import lightning.pytorch as L

In [2]:
print(torch.__version__)

2.6.0+cu124


In [3]:
train_transform = t.Compose(
    [
        t.AddFlanks(BarbadillaMartinezDataset.LEFT_FLANK, BarbadillaMartinezDataset.RIGHT_FLANK),
        t.RandomPadding(600),
        t.RightCrop(600, 600),
        t.Seq2Tensor(),
    ]
)
val_transform = t.Compose(
    [
        t.AddFlanks(BarbadillaMartinezDataset.LEFT_FLANK, BarbadillaMartinezDataset.RIGHT_FLANK),
        t.Padding(600),
        t.RightCrop(600, 600),
        t.Seq2Tensor(),
    ]
)


In [4]:
train_ds = BarbadillaMartinezDataset(split=[0,1,2,3], transform=train_transform, cell_type=["K562","HepG2"], root = "../data/")
val_ds = BarbadillaMartinezDataset(split=[4], transform=val_transform, cell_type=["K562","HepG2"], root = "../data/")
test_ds = BarbadillaMartinezDataset(split='test', transform=val_transform, cell_type=["K562","HepG2"], root = "../data/")

train_dl  = DataLoader(train_ds, batch_size=1024, num_workers=64, shuffle=True)
val_dl  = DataLoader(val_ds, batch_size=1024, num_workers=64, shuffle=False)


In [5]:
model = HumanLegNet(
        in_ch=4,
        output_dim=len(["K562","HepG2"]),
        stem_ch=64,
        stem_ks=11,
        ef_ks=9,
        ef_block_sizes=[80, 96, 112, 128],
        pool_sizes=[2, 2, 2, 2],
        resize_factor=4,
    )
seq_model = LitModel_BarbadillaMartinez(
    model=model, 
    cell_types=["K562", "HepG2"],
    #loss=nn.PoissonNLLLoss(log_input=False),
    loss=nn.MSELoss(),
    weight_decay=2e-1,
    lr=0.005, 
    print_each=1
)

In [6]:
trainer = L.Trainer(
        accelerator="gpu",
        devices=[0],
        max_epochs=1,
        gradient_clip_val=1,
        precision="16-mixed",
        enable_progress_bar=True,
        num_sanity_val_steps=0,
    )

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [7]:
trainer.fit(seq_model, train_dataloaders=train_dl, val_dataloaders=val_dl)

You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.291  

Epoch 0:   0%|          | 1/3012 [00:01<1:18:51,  0.64it/s, v_num=7]

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 0: 100%|██████████| 3012/3012 [12:30<00:00,  4.01it/s, v_num=7]
-------------------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 1.11050 
| Val Pearson K562: 0.84416 | Val Pearson HepG2: 0.78658 | Mean Val Pearson: 0.81537 |
| Train Pearson K562: 0.80946 | Train Pearson HepG2: 0.75488 | Mean Train Pearson: 0.78217 |
-------------------------------------------------------------------------------------------

Epoch 0: 100%|██████████| 3012/3012 [13:31<00:00,  3.71it/s, v_num=7, val_loss=1.110, val_K562_pearson=0.844, val_HepG2_pearson=0.787, val_pearson=0.815, train_loss=1.290]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 3012/3012 [13:32<00:00,  3.71it/s, v_num=7, val_loss=1.110, val_K562_pearson=0.844, val_HepG2_pearson=0.787, val_pearson=0.815, train_loss=1.290]


In [ ]:
test_dl  = DataLoader(test_ds, batch_size=1024, num_workers=64, shuffle=False)
answer = trainer.predict(seq_model, dataloaders=test_dl)
a = [f['predicted'] for f in answer]
b = torch.cat(a)
b = b.numpy()
target_columns = test_ds.target_columns
cols = target_columns + ['FEAT']
real = test_ds._data[cols].copy()
pred_columns = []
for ind, ta in enumerate(target_columns):
    c = f'{ta}_pred'
    real[c] = b[:, ind]
    pred_columns.append(c)
    
ag = real.groupby('FEAT')[pred_columns + target_columns].mean()
with open(f'perf_legnet_{test_ds.version}.txt', 'a') as inp:
    for r_name, p_name in zip(target_columns, pred_columns):
        p= pearsonr(ag[r_name], ag[p_name])
        print(r_name, p)
        print(r_name, p, file=inp)
        inp.flush()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 751/751 [01:06<00:00, 11.38it/s]
Log2RPM_K562 PearsonRResult(statistic=np.float64(0.9385254191684278), pvalue=np.float64(0.0))
Log2RPM_HepG2 PearsonRResult(statistic=np.float64(0.9153095607977082), pvalue=np.float64(0.0))
